# 01 — Point Representation

Before any algorithm, we need to establish one rule absolutely clearly. This is the rule that will silently break everything if you miss it:

```text
ipyleaflet click events  →  [lat, lon]   (latitude first)
GeoJSON coordinates      →  [lon, lat]   (longitude first)
```

They are backwards from each other. Always. Without exception.

If you get this wrong, your points will appear in the wrong ocean and your polygon tests will silently return false for everything. The bug is invisible until you look at the coordinates on a map — at which point you'll see your click near the coast of Somalia when you clicked on Iran.

## Setup

In [2]:
import json
from pathlib import Path
from ipyleaflet import Map, GeoJSON, Marker, basemaps
from ipywidgets import Output

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

print("Ready.")

Ready.


## The Coordinate Order Problem

Why do two tools in the same ecosystem use opposite conventions?

**ipyleaflet** follows Leaflet.js, which follows cartographic convention: latitude (north/south) first, longitude (east/west) second. This is how addresses are written, how GPS reads out, and how most mapping interfaces work.

**GeoJSON** follows mathematical convention: x-axis (longitude, east/west) first, y-axis (latitude, north/south) second. GeoJSON coordinates are `[x, y]` = `[lon, lat]`.

Both conventions are internally consistent. They just disagree with each other. And you are always in the middle.

```python
# ipyleaflet — kwargs['coordinates'] from a click:
coords = [35.695, 51.388]   # [lat, lon] — Tehran

# GeoJSON Point geometry:
geojson_point = {"type": "Point", "coordinates": [51.388, 35.695]}  # [lon, lat]

# Marker location:
marker = Marker(location=(35.695, 51.388))   # (lat, lon) — same as ipyleaflet
```

In [4]:
# Demonstrate the flip with Tehran's known coordinates
tehran_lat, tehran_lon = 35.695, 51.388

# What ipyleaflet click gives you:
click_coords = [tehran_lat, tehran_lon]   # [lat, lon]
print(f"From click event:  {click_coords}  (lat first)")

# What GeoJSON needs:
geojson_coords = [tehran_lon, tehran_lat]  # [lon, lat]
print(f"For GeoJSON:       {geojson_coords}  (lon first)")

# The extraction pattern — always the same:
lat = click_coords[0]
lon = click_coords[1]
print(f"\nExtracted:  lat={lat}  lon={lon}")

From click event:  [35.695, 51.388]  (lat first)
For GeoJSON:       [51.388, 35.695]  (lon first)

Extracted:  lat=35.695  lon=51.388


## Creating a GeoJSON Point Feature

Once we have a click, we want to store it as a proper GeoJSON Point feature — compatible with everything else in the course.

In [5]:
def click_to_point_feature(click_coords, properties=None):
    """
    Convert ipyleaflet click coordinates [lat, lon]
    to a GeoJSON Point Feature with [lon, lat] coordinates.
    """
    lat, lon = click_coords[0], click_coords[1]
    return {
        "type": "Feature",
        "properties": properties or {},
        "geometry": {
            "type": "Point",
            "coordinates": [lon, lat]   # GeoJSON: lon first
        }
    }

# Test with Tehran
fake_click = [35.695, 51.388]   # as if from kwargs['coordinates']
feature = click_to_point_feature(fake_click, properties={"label": "Tehran"})
print(json.dumps(feature, indent=2))

{
  "type": "Feature",
  "properties": {
    "label": "Tehran"
  },
  "geometry": {
    "type": "Point",
    "coordinates": [
      51.388,
      35.695
    ]
  }
}


## Adding a Marker at Click Location

`Marker` uses `(lat, lon)` — same order as ipyleaflet events — so no flip needed there. But `GeoJSON` and any polygon test needs `[lon, lat]`.

In [6]:
markers = []   # keep references so they can be removed
clicked_features = []

out = Output()
m = Map(center=(33, 44), zoom=4, basemap=basemaps.CartoDB.Positron)

def on_click(**kwargs):
    if kwargs.get('type') == 'click':
        coords = kwargs['coordinates']   # [lat, lon]
        lat, lon = coords[0], coords[1]

        # Add a Marker (uses lat, lon directly)
        marker = Marker(location=(lat, lon))
        m.add(marker)
        markers.append(marker)

        # Also store as GeoJSON feature (flip to lon, lat)
        feat = click_to_point_feature(coords, {"n": len(markers)})
        clicked_features.append(feat)

        with out:
            out.clear_output(wait=True)
            print(f"{len(markers)} marker(s) placed:")
            for f in clicked_features:
                c = f['geometry']['coordinates']
                print(f"  #{f['properties']['n']}  lon={c[0]:.4f}  lat={c[1]:.4f}  [GeoJSON order]")

m.on_interaction(on_click)
display(m, out)

Map(center=[33, 44], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_tex…

Output()

## Saving Clicked Points to a File

After clicking, we can persist the collected points as a GeoJSON FeatureCollection for use in later notebooks.

In [7]:
# Run after clicking the map above
if clicked_features:
    fc = {"type": "FeatureCollection", "features": clicked_features}
    out_path = DATA_DIR / "clicked_points.geojson"
    with open(out_path, "w") as f:
        json.dump(fc, f, indent=2)
    print(f"Saved {len(clicked_features)} point(s) → {out_path}")
else:
    print("No clicks yet — use the map above first, then re-run this cell.")

Saved 3 point(s) → data\clicked_points.geojson


## The Rule — Written Down

Put this somewhere visible and read it every time:

```python
# From a click event:
lat = kwargs['coordinates'][0]   # index 0 = lat
lon = kwargs['coordinates'][1]   # index 1 = lon

# For GeoJSON geometry:
"coordinates": [lon, lat]        # lon first

# For Marker location:
Marker(location=(lat, lon))      # lat first

# For polygon test (our PIP function, next notebook):
point_in_polygon(lon, lat, ring) # we'll use lon, lat consistently
```

The pattern: **always extract `lat, lon` from clicks immediately, then use `lon, lat` everywhere geometric**.

## Exercise A

Write a function `format_point(click_coords)` that takes ipyleaflet click coordinates and returns a GeoJSON Point Feature.

1. The function must include a `"click_index"` property that you pass in as a second argument.
2. Test it with these manually-specified fake click coordinates:
   - `[35.695, 51.388]` → Tehran
   - `[24.688, 46.675]` → Riyadh
   - `[40.417, -3.703]` → Madrid
3. Print the GeoJSON coordinates for each and verify the lon/lat order is correct.

In [9]:
import json

def format_point(click_coords, click_index):
    # click_coords is [lat, lon] from ipyleaflet
    # return a GeoJSON Feature with coordinates [lon, lat]
    # include click_index in properties
    pass  # your code here
    # click_coords is [lat, lon] from ipyleaflet
    lat, lon = click_coords[0], click_coords[1]
    
    return {
        "type": "Feature",
        "properties": {"click_index": click_index},
        "geometry": {
            "type": "Point",
            "coordinates": [lon, lat]  # The Flip: Lon first!
        }
    }

test_clicks = [
    ([35.695, 51.388], "Tehran"),
    ([24.688, 46.675], "Riyadh"),
    ([40.417, -3.703], "Madrid"),
]

for i, (coords, name) in enumerate(test_clicks):
    feat = format_point(coords, i + 1)
    # print the GeoJSON coords — should be [lon, lat]
    # Your code here
    gj_coords = feat['geometry']['coordinates']
    print(f"{name:8} | GeoJSON: {gj_coords}")

Tehran   | GeoJSON: [51.388, 35.695]
Riyadh   | GeoJSON: [46.675, 24.688]
Madrid   | GeoJSON: [-3.703, 40.417]


## Exercise B

Create a map that places a **Marker** at each click and simultaneously adds the click as a **GeoJSON Point** layer.

1. The Marker and the GeoJSON point must appear at the exact same location.
2. After 3 clicks, print the GeoJSON coordinates of all stored points.
3. Verify each point's `coordinates` field is in `[lon, lat]` order (longitude first).

In [10]:
from ipyleaflet import Map, GeoJSON, Marker, basemaps
from ipywidgets import Output

# Map + handler: place Marker AND GeoJSON point at each click
# After 3 clicks, print stored coordinates in [lon, lat] order
# Your code here
m_ex_b = Map(center=(30, 30), zoom=3, basemap=basemaps.CartoDB.Positron)
out_b = Output()
stored_gj_points = []

def dual_handler(**kwargs):
    if kwargs.get('type') == 'click':
        coords = kwargs['coordinates']  # [lat, lon]
        
        # 1. Marker (lat, lon)
        marker = Marker(location=coords)
        m_ex_b.add(marker)
        
        # 2. GeoJSON (flip to lon, lat)
        lat, lon = coords
        gj_feat = {
            "type": "Feature",
            "geometry": {"type": "Point", "coordinates": [lon, lat]}
        }
        stored_gj_points.append(gj_feat)
        m_ex_b.add(GeoJSON(data=gj_feat))
        
        with out_b:
            if len(stored_gj_points) >= 3:
                out_b.clear_output()
                print("Coordinates of stored GeoJSON points:")
                for p in stored_gj_points:
                    print(f"  {p['geometry']['coordinates']}")

m_ex_b.on_interaction(dual_handler)
display(m_ex_b, out_b)

Map(center=[30, 30], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_tex…

Output()

---

## Check Your Understanding

**1.** Why does ipyleaflet use `[lat, lon]` while GeoJSON uses `[lon, lat]`?
```python
# ipyleaflet (Leaflet): Uses cartographic tradition. In navigation and mapping, we typically state Latitude (North/South) before Longitude (East/West).

# GeoJSON: Uses mathematical/computer science tradition. On a 2D Cartesian plane, the X-axis (horizontal/Longitude) is always stated before the Y-axis (vertical/Latitude).
```
**2.** What would happen if you passed click coordinates directly to a GeoJSON geometry without flipping them?

```python
# If you pass [lat, lon] to GeoJSON, it interprets the Latitude as the X-coordinate and the Longitude as the Y-coordinate. Because Latitude only goes up to 90°, a Longitude of 120° would be an "out of bounds" Y-value. If they both fit within ranges (e.g., clicking in Europe), your point would simply appear in the wrong hemisphere—likely in the Southern Ocean or Antarctica—because the axes were swapped.
```

## Next

In [02 — Point in Polygon Basics](./02_Point_In_Polygon_Basics.ipynb), we build the intuition for what "inside a polygon" means before touching any algorithm — covering convex shapes, concave shapes, and the edge cases that cause headaches.